# 08b — Diagnósticos comparativos A vs B

Compara la medición A (línea de pobreza, `data/05_trayectoria_distancia`,
2016T2-2025T4) con la medición B (posición relativa, sin línea oficial,
`data/06_trayectoria_relativa`, 2003T3-2025T4): conteos de `par_valido_log`
y `par_valido_simetrica` (un filtro por objetivo, ver `.claude/decisiones.md`)
por partición train/embargo/holdout (mismo split que `06b`) y, solo para B,
la regresión a la media por decil separada por era.

**Lee** `data/05_trayectoria_distancia/`, `data/06_trayectoria_relativa/`.
**Escribe** solo CSV en `data/meta/`. No reconstruye ninguna capa.

In [ ]:
import sys
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import pandas as pd

import panel

TRAYECTORIA_A = REPO / "data" / "05_trayectoria_distancia"
TRAYECTORIA_B = REPO / "data" / "06_trayectoria_relativa"
META = REPO / "data" / "meta"
HOLDOUT_DESDE = (2023, 1)
FILTROS = ("par_valido_log", "par_valido_simetrica")

## Carga de pares A y B en memoria

Columnas mínimas de cada lado: la clave del par, los dos filtros
(`par_valido_log`/`par_valido_simetrica`, uno por objetivo -- A y B usan el
mismo criterio y los mismos nombres) y `era_t` (B, para separar historica/
regular).

In [ ]:
CLAVE_PAR = ["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th", "ANO4_t", "TRIMESTRE_t", "ANO4_th", "TRIMESTRE_th"]


def _cargar_capa2(base: Path, tipo: str, columnas: list[str]) -> pd.DataFrame:
    partes = []
    for p_anio in sorted((base / tipo).glob("ANO4=*")):
        for p_trim in sorted(p_anio.glob("TRIMESTRE=*")):
            p = p_trim / "data.parquet"
            if p.exists():
                df = pd.read_parquet(p)
                partes.append(df[[c for c in columnas if c in df.columns]])
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame(columns=columnas)


pares_a = {h: _cargar_capa2(TRAYECTORIA_A, f"pares_h{h}", CLAVE_PAR + list(FILTROS)) for h in (1, 4)}
pares_b = {h: _cargar_capa2(TRAYECTORIA_B, f"pares_h{h}", CLAVE_PAR + ["era_t", *FILTROS]) for h in (1, 4)}
for h in (1, 4):
    print(f"h={h}: A={len(pares_a[h]):,} pares (2016+), B={len(pares_b[h]):,} pares (todas las eras)")

## Conteos comparativos A vs B, por filtro, partición y era

Partición vía `panel.particion_train_holdout` (mismo split que `06b`:
holdout = origen `t>=2023T1`, train = destino `t+h<=2022T4`, embargo en el
medio). El holdout debe ser el mismo universo en A y B -- A es un
subconjunto temporal de B; lo que cambia es el train (B suma 2003-2015).

In [ ]:
filas_conteo = []
for h in (1, 4):
    a, b = pares_a[h], pares_b[h]
    particion_a = panel.particion_train_holdout(a, HOLDOUT_DESDE)
    particion_b = panel.particion_train_holdout(b, HOLDOUT_DESDE)
    for filtro in FILTROS:
        for nombre_particion in ("train", "embargo", "holdout"):
            filas_conteo.append({
                "h": h, "filtro": filtro, "medicion": "A", "era": "regular", "particion": nombre_particion,
                "n_par_valido": int((particion_a.eq(nombre_particion) & a[filtro]).sum()),
            })
            for era, grupo_b in b.groupby("era_t"):
                particion_b_era = particion_b[grupo_b.index]
                filas_conteo.append({
                    "h": h, "filtro": filtro, "medicion": "B", "era": era, "particion": nombre_particion,
                    "n_par_valido": int((particion_b_era.eq(nombre_particion) & grupo_b[filtro]).sum()),
                })

comparacion_ab = pd.DataFrame(filas_conteo)
comparacion_ab.to_csv(META / "comparacion_ab_conteos_particion.csv", index=False)
comparacion_ab.pivot_table(index=["filtro", "h", "particion"], columns=["medicion", "era"], values="n_par_valido")

### Chequeo: en el solape 2016+, A y B deben dar el mismo conteo por filtro

`par_valido_log` exige `ITF`/`y_ae > 0` en ambas puntas en los dos lados --
se espera igualdad exacta (ya confirmada vía la identidad algebraica de
`tests/test_puente_medicion_b.py`); si difiere, se frena acá.
`par_valido_simetrica` admite `ITF=0`/`y_ae=0`, pero normaliza distinto (A
por la línea de pobreza, B por la mediana) -- puede haber diferencias
legítimas; se reportan sin frenar, con la causa.

In [ ]:
def _diagnostico_discrepancia(a: pd.DataFrame, particion_a: pd.Series, b_regular: pd.DataFrame,
                               particion_b_regular: pd.Series, filtro: str, nombre_particion: str) -> pd.DataFrame:
    m = a.assign(particion=particion_a).merge(
        b_regular.assign(particion=particion_b_regular), on=CLAVE_PAR, how="outer", suffixes=("_a", "_b"))
    en_la_particion = (m["particion_a"] == nombre_particion) | (m["particion_b"] == nombre_particion)
    discrepantes = m[en_la_particion & (m[f"{filtro}_a"].fillna(False) != m[f"{filtro}_b"].fillna(False))]
    return discrepantes


for h in (1, 4):
    a, b = pares_a[h], pares_b[h]
    particion_a = panel.particion_train_holdout(a, HOLDOUT_DESDE)
    b_regular = b[b["era_t"] == "regular"]
    particion_b_regular = panel.particion_train_holdout(b_regular, HOLDOUT_DESDE)

    for filtro in FILTROS:
        for nombre_particion in ("train", "embargo", "holdout"):
            n_a = int((particion_a.eq(nombre_particion) & a[filtro]).sum())
            n_b = int((particion_b_regular.eq(nombre_particion) & b_regular[filtro]).sum())
            if n_a == n_b:
                print(f"h={h} {filtro} {nombre_particion}: A=B={n_a} (coinciden)")
                continue
            discrepantes = _diagnostico_discrepancia(a, particion_a, b_regular, particion_b_regular,
                                                       filtro, nombre_particion)
            if filtro == "par_valido_log":
                raise AssertionError(
                    f"h={h}, particion={nombre_particion}: par_valido_log A={n_a} != B={n_b} "
                    f"-- no se esperaba ninguna diferencia (ver test_puente_medicion_b.py). "
                    f"{len(discrepantes)} pares discrepantes:\n{discrepantes.head(20).to_string()}"
                )
            print(f"h={h} {filtro} {nombre_particion}: A={n_a} B={n_b} (diferencia={n_a - n_b}) -- "
                  f"{len(discrepantes)} pares discrepantes, no se frena (ver celda de causa abajo)")

#### Causa de la diferencia en `par_valido_simetrica`

`ratio_lp` (A) depende de la canasta regional (`REGION`/`cbt_ae`); si para
algún hogar esa canasta no está disponible, `ratio_lp` -- y por lo tanto
`variacion_simetrica` -- da `NA` sin que haya no respuesta ni problema en
`y_ae`. `posicion_relativa_aglomerado` (B) no depende de ninguna canasta,
así que esos casos quedan válidos en B y no en A -- es la dirección opuesta
a la que vimos con `par_valido_log` e `ITF=0`, y es una diferencia legítima
de diseño, no un error.

In [ ]:
h_ejemplo = 1
a, b = pares_a[h_ejemplo], pares_b[h_ejemplo]
b_regular = b[b["era_t"] == "regular"]
particion_a = panel.particion_train_holdout(a, HOLDOUT_DESDE)
particion_b_regular = panel.particion_train_holdout(b_regular, HOLDOUT_DESDE)
discrepantes = _diagnostico_discrepancia(a, particion_a, b_regular, particion_b_regular,
                                          "par_valido_simetrica", "train")
print(f"h={h_ejemplo}, train: {len(discrepantes)} pares con par_valido_simetrica distinto entre A y B")
print(discrepantes[["par_valido_simetrica_a", "par_valido_simetrica_b"]].value_counts())

## Diagnóstico descriptivo de B: regresión a la media por decil, por era

Mismo patrón que la sección 2 de `06b` (línea base de reversión a la
media), pero sobre `posicion_relativa_aglomerado_t` (B) y separado por
`era_t` además de por `{todos, par_valido_log}` -- puramente descriptivo,
sin umbral de aprobación. Se usa `par_valido_log` (no `par_valido_simetrica`)
porque es el filtro del objetivo principal de B (`variacion_propia_b`).

In [ ]:
COLUMNAS_DECIL = CLAVE_PAR + ["era_t", "posicion_relativa_aglomerado_t", "peso_hogar_t", "par_valido_log"]
pares_decil = {h: _cargar_capa2(TRAYECTORIA_B, f"pares_h{h}", COLUMNAS_DECIL) for h in (1, 4)}


def _regresion_media_decil(df: pd.DataFrame) -> pd.DataFrame:
    valido = df["posicion_relativa_aglomerado_t"].notna() & (df["posicion_relativa_aglomerado_t"] > 0)
    df = df[valido].copy()
    df["decil"] = pd.qcut(df["posicion_relativa_aglomerado_t"], 10, labels=False, duplicates="drop") + 1
    return df.groupby("decil").agg(
        n_pares=("CODUSU", "size"),
        posicion_t_mediana=("posicion_relativa_aglomerado_t", "median"),
    ).reset_index()


tablas_decil = {}
for h in (1, 4):
    df = pares_decil[h]
    for era in ("historica", "regular"):
        for etiqueta, subconjunto in (("todos", df), ("par_valido", df[df["par_valido_log"]])):
            clave = (h, etiqueta, era)
            tabla = _regresion_media_decil(subconjunto[subconjunto["era_t"] == era])
            tablas_decil[clave] = tabla
            tabla.to_csv(META / f"trayectoria_relativa_regresion_media_decil_h{h}_{etiqueta}_{era}.csv", index=False)

print(f"{len(tablas_decil)} tablas escritas en data/meta/trayectoria_relativa_regresion_media_decil_*.csv")

### ¿El patrón es el mismo entre eras?

Mediana de `posicion_relativa_aglomerado_t` por decil, histórica vs.
regular, `h=1` y `par_valido_log` -- si el patrón de reversión a la media
difiere marcado entre eras, es una señal a documentar, no a promediar.

In [ ]:
comparacion_eras = tablas_decil[(1, "par_valido", "historica")][["decil", "posicion_t_mediana"]].rename(
    columns={"posicion_t_mediana": "historica"}).merge(
    tablas_decil[(1, "par_valido", "regular")][["decil", "posicion_t_mediana"]].rename(
        columns={"posicion_t_mediana": "regular"}), on="decil")
comparacion_eras["diferencia"] = comparacion_eras["historica"] - comparacion_eras["regular"]
comparacion_eras

## 6. Conteos de B por era + entrenamiento total A vs. B

`comparacion_ab_conteos_particion.csv` (sección 4) ya tiene el desagregado
por era -- esta sección solo lo presenta en una tabla ancha dedicada y
agrega el total de entrenamiento (B = histórica + regular) contra A, con el
factor de aumento.

In [ ]:
conteos_b_por_era = comparacion_ab[comparacion_ab["medicion"] == "B"].pivot_table(
    index=["filtro", "h", "particion"], columns="era", values="n_par_valido", fill_value=0)
conteos_b_por_era.to_csv(META / "trayectoria_relativa_conteos_por_era.csv")
conteos_b_por_era

In [ ]:
entrenamiento = comparacion_ab[comparacion_ab["particion"] == "train"]
train_a = entrenamiento[entrenamiento["medicion"] == "A"].set_index(["filtro", "h"])["n_par_valido"]
train_b_total = entrenamiento[entrenamiento["medicion"] == "B"].groupby(["filtro", "h"])["n_par_valido"].sum()

resumen_entrenamiento = pd.DataFrame({"train_A": train_a, "train_B_total": train_b_total})
resumen_entrenamiento["factor_aumento"] = resumen_entrenamiento["train_B_total"] / resumen_entrenamiento["train_A"]
resumen_entrenamiento.to_csv(META / "comparacion_ab_entrenamiento_total.csv")
resumen_entrenamiento

## 7. Variación propia y variación log promedio por decil de posición en `t`, por era

Aclaración a la sección 5: los valores de ahí (`posicion_t_mediana`) eran
la posición relativa típica del decil, no la variación. Acá: media de
`variacion_propia_b` y `variacion_log_b` por decil de
`posicion_relativa_aglomerado_t`, filtrado a `par_valido_log` (el filtro
del objetivo principal), separado por era -- deciles calculados dentro de
cada era por separado, igual que en la sección 5.

In [ ]:
import numpy as np

COLUMNAS_VARIACION = CLAVE_PAR + ["era_t", "posicion_relativa_aglomerado_t", "par_valido_log",
                                   "variacion_propia_b", "variacion_log_b"]
pares_variacion = {h: _cargar_capa2(TRAYECTORIA_B, f"pares_h{h}", COLUMNAS_VARIACION) for h in (1, 4)}


def _variacion_por_decil(df: pd.DataFrame) -> pd.DataFrame:
    df = df[df["par_valido_log"]].copy()
    df["decil"] = pd.qcut(df["posicion_relativa_aglomerado_t"], 10, labels=False, duplicates="drop") + 1
    return df.groupby("decil").agg(
        n_pares=("CODUSU", "size"),
        posicion_t_mediana=("posicion_relativa_aglomerado_t", "median"),
        variacion_propia_b_media=("variacion_propia_b", "mean"),
        variacion_log_b_media=("variacion_log_b", "mean"),
    ).reset_index()


tablas_variacion = {}
for h in (1, 4):
    df = pares_variacion[h]
    for era in ("historica", "regular"):
        tabla = _variacion_por_decil(df[df["era_t"] == era])
        tablas_variacion[(h, era)] = tabla
        tabla.to_csv(META / f"trayectoria_relativa_variacion_media_decil_h{h}_{era}.csv", index=False)

print(f"{len(tablas_variacion)} tablas escritas en data/meta/trayectoria_relativa_variacion_media_decil_*.csv")

### Lado a lado, histórica vs. regular, con diferencia por decil

In [ ]:
def _lado_a_lado(h: int) -> pd.DataFrame:
    hist = tablas_variacion[(h, "historica")].set_index("decil")
    reg = tablas_variacion[(h, "regular")].set_index("decil")
    return pd.DataFrame({
        "posicion_t_mediana_historica": hist["posicion_t_mediana"],
        "posicion_t_mediana_regular": reg["posicion_t_mediana"],
        "variacion_propia_b_historica": hist["variacion_propia_b_media"],
        "variacion_propia_b_regular": reg["variacion_propia_b_media"],
        "variacion_propia_b_diferencia": hist["variacion_propia_b_media"] - reg["variacion_propia_b_media"],
        "variacion_log_b_historica": hist["variacion_log_b_media"],
        "variacion_log_b_regular": reg["variacion_log_b_media"],
        "variacion_log_b_diferencia": hist["variacion_log_b_media"] - reg["variacion_log_b_media"],
    })


comparacion_variacion = {h: _lado_a_lado(h) for h in (1, 4)}
for h in (1, 4):
    comparacion_variacion[h].to_csv(META / f"trayectoria_relativa_variacion_media_decil_comparacion_h{h}.csv")
print("h=1:")
display(comparacion_variacion[1])
print("h=4:")
display(comparacion_variacion[4])

### Pendiente simple de reversión a la media, por era

`variacion ~ log(posicion_relativa_aglomerado_t)`, OLS sin ponderar
(`np.polyfit`, solo descriptivo) -- una pendiente más negativa es más
reversión a la media (posiciones altas en `t` asociadas a mayor caída
propia hacia `t+h`).

In [ ]:
def _pendiente(df: pd.DataFrame, columna_y: str) -> float:
    x = np.log(df["posicion_relativa_aglomerado_t"].astype(float))
    y = df[columna_y].astype(float)
    valido = x.notna() & y.notna()
    pendiente, _ordenada = np.polyfit(x[valido], y[valido], 1)
    return pendiente


filas_pendiente = []
for h in (1, 4):
    df = pares_variacion[h]
    for era in ("historica", "regular"):
        subconjunto = df[(df["era_t"] == era) & df["par_valido_log"]]
        filas_pendiente.append({
            "h": h, "era": era, "n_pares": len(subconjunto),
            "pendiente_variacion_propia_b": _pendiente(subconjunto, "variacion_propia_b"),
            "pendiente_variacion_log_b": _pendiente(subconjunto, "variacion_log_b"),
        })

pendientes = pd.DataFrame(filas_pendiente)
pendientes_anchas = pendientes.pivot(index="h", columns="era",
                                      values=["pendiente_variacion_propia_b", "pendiente_variacion_log_b"])
pendientes_anchas["dif_propia_hist_menos_reg"] = (
    pendientes_anchas[("pendiente_variacion_propia_b", "historica")]
    - pendientes_anchas[("pendiente_variacion_propia_b", "regular")])
pendientes_anchas["dif_log_hist_menos_reg"] = (
    pendientes_anchas[("pendiente_variacion_log_b", "historica")]
    - pendientes_anchas[("pendiente_variacion_log_b", "regular")])

pendientes.to_csv(META / "trayectoria_relativa_pendiente_reversion_media.csv", index=False)
pendientes_anchas

---

## Cierre

Conteos comparativos A vs B por filtro/partición/era (sección 4), conteos
de B desagregados por era y entrenamiento total A vs. B con factor de
aumento (sección 6), y variación propia/log promedio por decil de posición
en `t` -- lado a lado histórica/regular, con pendiente de reversión a la
media (sección 7). Todas las salidas en `data/meta/comparacion_ab_*.csv` y
`data/meta/trayectoria_relativa_*.csv`.